# CIFAR-10 ResNet Training (224×224)
- Loads from `cifar10_224_preprocessed.npz`
- 15K images (10K + 5K augmented) at 224×224
- Bicubic Upscale + Denoise + CLAHE + Sharpen + Augmentation
- **ResNet-style CNN** with residual skip connections
- 50 Epochs with LR Scheduler

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import time

## 1. Setup Device

In [ ]:
device = torch.device('mps' if torch.backends.mps.is_available() else 
                       'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 2. Load Preprocessed Data

In [ ]:
data = np.load('cifar10_224_preprocessed.npz')

X_train = data['X_train'].astype(np.float32)
X_val = data['X_val'].astype(np.float32)
X_test = data['X_test'].astype(np.float32)
y_train = data['y_train_int'].astype(np.int64)
y_val = data['y_val_int'].astype(np.int64)
y_test = data['y_test_int'].astype(np.int64)
label_names = list(data['label_names'])

# Convert (N, H, W, C) -> (N, C, H, W) for PyTorch
X_train = X_train.transpose(0, 3, 1, 2)
X_val = X_val.transpose(0, 3, 1, 2)
X_test = X_test.transpose(0, 3, 1, 2)

print(f'Train: {X_train.shape} | Labels: {y_train.shape}')
print(f'Val:   {X_val.shape} | Labels: {y_val.shape}')
print(f'Test:  {X_test.shape} | Labels: {y_test.shape}')
print(f'Classes: {label_names}')

## 3. Create DataLoaders

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train), torch.tensor(y_train)),
    batch_size=BATCH_SIZE, shuffle=True
)
val_loader = DataLoader(
    TensorDataset(torch.tensor(X_val), torch.tensor(y_val)),
    batch_size=64
)
test_loader = DataLoader(
    TensorDataset(torch.tensor(X_test), torch.tensor(y_test)),
    batch_size=64
)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')
print(f'Test batches:  {len(test_loader)}')

## 4. Define ResNet Model
ResNet uses **residual/skip connections** — the input to a block is added back to its output.  
This lets gradients flow directly through the network, enabling deeper models without vanishing gradients.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class ResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # Stem: 224x224 -> 56x56
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(3, stride=2, padding=1)
        )

        # Residual layers
        self.layer1 = self._make_layer(64, 64, 2, stride=1)    # 56x56
        self.layer2 = self._make_layer(64, 128, 2, stride=2)   # 28x28
        self.layer3 = self._make_layer(128, 256, 2, stride=2)  # 14x14
        self.layer4 = self._make_layer(256, 512, 2, stride=2)  # 7x7

        # Classifier
        self.avgpool = nn.AdaptiveAvgPool2d(1)   # 7x7 -> 1x1
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, in_ch, out_ch, num_blocks, stride):
        layers = [ResidualBlock(in_ch, out_ch, stride)]
        for _ in range(1, num_blocks):
            layers.append(ResidualBlock(out_ch, out_ch))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x


model = ResNet(num_classes=10).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Total Parameters: {total_params:,}')
print(model)

## 5. Loss, Optimizer & Scheduler

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

print('Optimizer: Adam (lr=0.001, weight_decay=1e-4)')
print('Scheduler: ReduceLROnPlateau (factor=0.5, patience=5)')

## 6. Training Loop (50 Epochs)

In [ ]:
EPOCHS = 50
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0

print(f'Training ResNet for {EPOCHS} epochs on {len(X_train):,} images (224x224)...')
print(f'{"="*65}')

start = time.time()

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
        total += xb.size(0)
    train_loss = running_loss / total
    train_acc = correct / total

    # --- Validate ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            val_loss += criterion(out, yb).item() * xb.size(0)
            val_correct += (out.argmax(1) == yb).sum().item()
            val_total += xb.size(0)
    val_loss /= val_total
    val_acc = val_correct / val_total

    # LR Scheduler
    old_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_loss)
    new_lr = optimizer.param_groups[0]['lr']
    if new_lr != old_lr:
        print(f'  >> LR reduced: {old_lr:.6f} -> {new_lr:.6f}')

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_resnet_224.pth')

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:2d}/{EPOCHS} | '
              f'Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | '
              f'Val Loss: {val_loss:.4f} Acc: {val_acc*100:.2f}% | '
              f'Best: {best_val_acc*100:.2f}%')

elapsed = time.time() - start
print(f'{"="*65}')
print(f'Training completed in {elapsed:.1f}s')
print(f'Best Validation Accuracy: {best_val_acc*100:.2f}%')

## 7. Test Set Evaluation

In [ ]:
model.load_state_dict(torch.load('best_resnet_224.pth', weights_only=True))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        all_preds.append(model(xb).argmax(1).cpu().numpy())
        all_labels.append(yb.numpy())

y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_labels)
test_acc = (y_pred == y_true).mean()

print(f'Test Accuracy: {test_acc*100:.2f}%')
print(f'\nPer-Class Accuracy:')
print('-' * 30)
for i, name in enumerate(label_names):
    mask = y_true == i
    if mask.sum() > 0:
        acc = (y_pred[mask] == i).mean()
        print(f'  {name:<12s}: {acc*100:.1f}%')

## 8. Classification Report

In [ ]:
print(classification_report(y_true, y_pred, target_names=label_names))

## 9. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, EPOCHS + 1)

ax1.plot(epochs_range, history['train_loss'], 'b-', label='Train Loss', alpha=0.8)
ax1.plot(epochs_range, history['val_loss'], 'r-', label='Val Loss', alpha=0.8)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss Over Epochs')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history['train_acc'], 'b-', label='Train Acc', alpha=0.8)
ax2.plot(epochs_range, history['val_acc'], 'r-', label='Val Acc', alpha=0.8)
ax2.axhline(y=test_acc, color='green', linestyle='--', label=f'Test: {test_acc*100:.2f}%')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy Over Epochs')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle(f'ResNet 224\u00d7224 (Upscale + Denoise + CLAHE + Sharp + Aug) \u2014 Test: {test_acc*100:.2f}%',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves_resnet.png', dpi=150)
plt.show()

## 10. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(label_names, rotation=45, ha='right')
ax.set_yticklabels(label_names)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix (ResNet 224\u00d7224) \u2014 Test Accuracy: {test_acc*100:.2f}%')

for i in range(10):
    for j in range(10):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color=color, fontsize=9)

plt.colorbar(im)
plt.tight_layout()
plt.savefig('confusion_matrix_resnet.png', dpi=150)
plt.show()

## 11. Sample Predictions

In [ ]:
test_display = X_test.transpose(0, 2, 3, 1)
test_display = (test_display - test_display.min()) / (test_display.max() - test_display.min())

np.random.seed(42)
indices = np.random.choice(len(y_true), 20, replace=False)

fig, axes = plt.subplots(2, 10, figsize=(18, 4.5))
for i, idx in enumerate(indices):
    ax = axes[i // 10, i % 10]
    ax.imshow(test_display[idx])
    true_label = label_names[y_true[idx]]
    pred_label = label_names[y_pred[idx]]
    color = 'green' if y_true[idx] == y_pred[idx] else 'red'
    ax.set_title(f'T: {true_label}\nP: {pred_label}', fontsize=8, color=color)
    ax.axis('off')

plt.suptitle('ResNet Sample Predictions (Green=Correct, Red=Wrong)', fontsize=13)
plt.tight_layout()
plt.savefig('sample_predictions_resnet.png', dpi=150)
plt.show()

## Summary

| Setting | Value |
|---|---|
| Data | `cifar10_224_preprocessed.npz` |
| Total Images | 15K (10K + 5K augmented) |
| Preprocessing | Bicubic Upscale + Denoise + CLAHE + Sharpen |
| Augmentation | Flip + Rotation + Crop + Brightness + Cutout |
| Resolution | 224×224 (upscaled from 32×32) |
| Split | 12K train / 3K val / 2K test |
| Model | **ResNet** (4 residual layers, 8 res blocks) |
| Key Feature | **Skip connections** (input + output) |
| Parameters | ~11.2M |
| Epochs | 50 |
| Batch Size | 32 (train) / 64 (val/test) |
| Optimizer | Adam (lr=0.001, weight_decay=1e-4) |
| Scheduler | ReduceLROnPlateau |